## Code Flow

1. Load the Dataset

2. Basic Preprocessing

3. Training Process
    - Create the Model
    - Forward Pass
    - Loss Calculation
    - Backpropagation
    - Parameters Update

4. Model Evaluation

In [1]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

In [3]:
df.shape

(569, 33)

In [4]:
df.drop(columns=['id','Unnamed: 32'], inplace=True)

In [5]:
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,radius_se,texture_se,perimeter_se,area_se,smoothness_se,compactness_se,concavity_se,concave points_se,symmetry_se,fractal_dimension_se,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,0.4956,1.1560,3.445,27.23,0.009110,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.011490,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [6]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['diagnosis']), df['diagnosis'], test_size=0.2, random_state=42)

In [7]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [8]:
print(X_train.shape)
print(X_test.shape)

(455, 30)
(114, 30)


In [9]:
print(y_train.shape)
print(y_test.shape)

(455,)
(114,)


In [10]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [11]:
y_train[:5], y_test[:5]

(array([0, 1, 0, 0, 0]), array([0, 1, 1, 0, 0]))

In [23]:
# Convert to PyTorch tensors
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [24]:
print(X_train_tensor.shape)
print(X_test_tensor.shape)
print(y_train_tensor.shape)
print(y_test_tensor.shape)

torch.Size([455, 30])
torch.Size([114, 30])
torch.Size([455])
torch.Size([114])


In [25]:
# Define the model
class SimpleNN():
    def __init__(self, X):
        self.input_size = X.shape[1]

        self.weights = torch.randn(self.input_size, 1, requires_grad=True, dtype=torch.float64)
        self.bias = torch.randn(1, requires_grad=True, dtype=torch.float64)

    def forward(self, X):
        z = torch.matmul(X, self.weights) + self.bias
        return torch.sigmoid(z)
    
    def loss(self, y_pred, y):
        # Binary Cross-Entropy Loss
        # Clamp to avoid log(0)
        epsilon = 1e-7
        y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)
        return -torch.mean(y * torch.log(y_pred) + (1 - y) * torch.log(1 - y_pred))

In [26]:
learning_rate = 0.01
epochs = 100

## Training Pipeline

In [28]:
# Initialize the model
model = SimpleNN(X_train_tensor)

# Training loop
for epoch in range(epochs):
    # 1. Forward pass
    y_pred = model.forward(X_train_tensor)

    # 2. Compute loss
    loss = model.loss(y_pred, y_train_tensor.float().view(-1, 1))

    # 3. Backward pass
    loss.backward()

    # 4. Update weights and bias
    with torch.no_grad(): # No need to track gradients for the update step
        # Update weights and bias
        model.weights -= learning_rate * model.weights.grad
        model.bias -= learning_rate * model.bias.grad

        # Zero the gradients after updating
        model.weights.grad.zero_()
        model.bias.grad.zero_()

    # Print loss every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}')

Epoch [10/100], Loss: 0.3250
Epoch [20/100], Loss: 0.3178
Epoch [30/100], Loss: 0.3112
Epoch [40/100], Loss: 0.3050
Epoch [50/100], Loss: 0.2993
Epoch [60/100], Loss: 0.2938
Epoch [70/100], Loss: 0.2887
Epoch [80/100], Loss: 0.2839
Epoch [90/100], Loss: 0.2794
Epoch [100/100], Loss: 0.2750


In [29]:
print(f'Final Loss: {loss.item():.4f}')

Final Loss: 0.2750


In [ ]:
print(model.weights)
print(model.bias)

tensor([[ 1.1539],
        [-1.2635],
        [-0.3592],
        [ 3.1030],
        [-1.4365],
        [ 0.0259],
        [ 0.2207],
        [ 0.7687],
        [-0.2311],
        [-1.3847],
        [ 1.1822],
        [-0.4287],
        [ 1.6570],
        [-0.6107],
        [-0.1275],
        [-0.0679],
        [-1.1659],
        [-0.1846],
        [ 1.0457],
        [-1.0537],
        [-0.2876],
        [ 0.4195],
        [-0.0927],
        [ 0.2673],
        [ 0.4419],
        [ 1.4317],
        [ 0.5680],
        [ 1.6793],
        [ 0.3591],
        [ 0.3247]], dtype=torch.float64, requires_grad=True)
tensor([-1.7994], dtype=torch.float64, requires_grad=True)


In [31]:
# Evaluate the model
with torch.no_grad():
    y_test_pred = model.forward(X_test_tensor)
    y_test_pred = (y_test_pred > 0.5).float()
    accuracy = (y_test_pred.view(-1) == y_test_tensor).float().mean()
    print(f'Accuracy: {accuracy.item() * 100:.2f}%')

Accuracy: 92.98%
